# 03d Fine-tuning

Warm-start fine-tuning with Berlin champions on Leipzig data.

**Inputs:**
- Berlin champions (from 03b)
- Leipzig finetune/test data (from 03a)
- Transfer evaluation (from 03c)

**Outputs:**
- `finetuning_curve.json` - Learning curves for ML and NN
- McNemar significance tests
- Power-law curve fitting and 95% recovery extrapolation
- Per-genus recovery analysis

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import pickle
import time
import datetime
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

start_time = time.time()
config = load_experiment_config()
print(f"Fine-tuning started")

In [ ]:
# Step 1: Load models and data
print(f"\n{'='*60}")
print(f"STEP 1: Load Models and Data")
print(f"{'='*60}")

model_dir = OUTPUT_DIR / 'models'

# Load ML champion
ml_model = training.load_model(model_dir / 'berlin_ml_champion.pkl')
ml_metadata = json.loads((model_dir / 'berlin_ml_champion.pkl.metadata.json').read_text())
ml_name = ml_metadata['model_name']

# Load NN champion
nn_model = training.load_model(model_dir / 'berlin_nn_champion.pt')
nn_metadata = json.loads((model_dir / 'berlin_nn_champion.pt.metadata.json').read_text())
nn_name = nn_metadata['model_name']

print(f"ML Champion: {ml_name}")
print(f"NN Champion: {nn_name}")

# Load Leipzig data
leipzig_finetune, leipzig_test = data_loading.load_leipzig_splits(DATA_DIR / 'phase_3_experiments')

# Load label encoder and scalers
with (model_dir / 'label_encoder.pkl').open('rb') as f:
    label_encoder = pickle.load(f)
    label_to_idx = label_encoder['label_to_idx']
    idx_to_label = label_encoder['idx_to_label']

with (model_dir / 'berlin_scaler.pkl').open('rb') as f:
    berlin_scaler = pickle.load(f)

feature_cols = ml_metadata['feature_columns']
class_labels = list(idx_to_label.values())

# Prepare test set (scaled with Berlin scaler for transfer)
x_test = leipzig_test[feature_cols].to_numpy(dtype=float)
y_test = leipzig_test['genus_latin'].map(label_to_idx).to_numpy()
x_test_scaled = berlin_scaler.transform(x_test)

print(f"\nLeipzig finetune: {len(leipzig_finetune)} samples")
print(f"Leipzig test: {len(leipzig_test)} samples")
print(f"Features: {len(feature_cols)}")
print(f"Genera: {len(class_labels)}")

In [ ]:
# Step 2: ML Fine-tuning with stratified subsets
print(f"\n{'='*60}")
print(f"STEP 2: ML Fine-Tuning (Warm-Start XGBoost)")
print(f"{'='*60}")

fractions = config['finetuning']['fractions']
ml_results = []

for frac in fractions:
    print(f"\nFine-tuning with {frac:.1%} of Leipzig data...")
    
    # Create stratified subset
    subsets = training.create_stratified_subsets(
        leipzig_finetune,
        fractions=[frac],
        label_col='genus_latin',
        random_seed=config['global']['random_seed'],
    )
    finetune_subset = subsets[frac]
    
    # Prepare data (scaled with Berlin scaler)
    x_finetune = finetune_subset[feature_cols].to_numpy(dtype=float)
    y_finetune = finetune_subset['genus_latin'].map(label_to_idx).to_numpy()
    x_finetune_scaled = berlin_scaler.transform(x_finetune)
    
    # Warm-start fine-tuning
    n_estimators = config['finetuning']['ml_warm_start_estimators']
    finetuned_model = training.finetune_xgboost(
        ml_model,
        x_finetune_scaled,
        y_finetune,
        n_new_estimators=n_estimators,
    )
    
    # Evaluate
    preds = finetuned_model.predict(x_test_scaled)
    metrics = evaluation.compute_metrics(y_test, preds)
    
    ml_results.append({
        'fraction': frac,
        'n_samples': len(finetune_subset),
        'f1_score': metrics['f1_score'],
        'accuracy': metrics['accuracy'],
        'predictions': preds.tolist(),
    })
    
    print(f"  F1: {metrics['f1_score']:.4f}")
    print(f"  Samples: {len(finetune_subset)}")

In [ ]:
# Step 3: NN Fine-tuning
print(f"\n{'='*60}")
print(f"STEP 3: NN Fine-Tuning (Warm-Start CNN1D)")
print(f"{'='*60}")

nn_results = []

for frac in fractions:
    print(f"\nFine-tuning with {frac:.1%} of Leipzig data...")
    
    # Create stratified subset
    subsets = training.create_stratified_subsets(
        leipzig_finetune,
        fractions=[frac],
        label_col='genus_latin',
        random_seed=config['global']['random_seed'],
    )
    finetune_subset = subsets[frac]
    
    # Prepare data
    x_finetune = finetune_subset[feature_cols].to_numpy(dtype=float)
    y_finetune = finetune_subset['genus_latin'].map(label_to_idx).to_numpy()
    x_finetune_scaled = berlin_scaler.transform(x_finetune)
    
    # Warm-start fine-tuning
    finetuned_model = training.finetune_neural_network(
        nn_model,
        x_finetune_scaled,
        y_finetune,
        x_val=x_test_scaled,
        y_val=y_test,
        epochs=config['finetuning']['nn_epochs'],
        batch_size=config['finetuning']['nn_batch_size'],
    )
    
    # Evaluate
    preds = finetuned_model.predict(x_test_scaled)
    metrics = evaluation.compute_metrics(y_test, preds)
    
    nn_results.append({
        'fraction': frac,
        'n_samples': len(finetune_subset),
        'f1_score': metrics['f1_score'],
        'accuracy': metrics['accuracy'],
        'predictions': preds.tolist(),
    })
    
    print(f"  F1: {metrics['f1_score']:.4f}")
    print(f"  Samples: {len(finetune_subset)}")

In [ ]:
# Step 4: From-scratch baselines (Leipzig-specific scaler)
print(f"\n{'='*60}")
print(f"STEP 4: From-Scratch Baselines")
print(f"{'='*60}")

ml_baseline_results = []

for frac in fractions:
    print(f"\nTraining from scratch with {frac:.1%} of Leipzig data...")
    
    # Create stratified subset
    subsets = training.create_stratified_subsets(
        leipzig_finetune,
        fractions=[frac],
        label_col='genus_latin',
        random_seed=config['global']['random_seed'],
    )
    train_subset = subsets[frac]
    
    # Leipzig-specific scaler
    x_train = train_subset[feature_cols].to_numpy(dtype=float)
    y_train = train_subset['genus_latin'].map(label_to_idx).to_numpy()
    
    leipzig_scaler = StandardScaler()
    x_train_scaled = leipzig_scaler.fit_transform(x_train)
    x_test_leipzig_scaled = leipzig_scaler.transform(x_test)
    
    # Train from scratch with same hyperparameters
    baseline_params = ml_metadata.get('best_params', {})
    baseline_model = models.create_model(ml_name, model_params=baseline_params)
    baseline_model.fit(x_train_scaled, y_train)
    
    # Evaluate
    preds = baseline_model.predict(x_test_leipzig_scaled)
    metrics = evaluation.compute_metrics(y_test, preds)
    
    ml_baseline_results.append({
        'fraction': frac,
        'n_samples': len(train_subset),
        'f1_score': metrics['f1_score'],
        'accuracy': metrics['accuracy'],
        'predictions': preds.tolist(),
    })
    
    print(f"  F1: {metrics['f1_score']:.4f}")

print(f"\nFrom-scratch baseline complete")

In [ ]:
# Step 5: McNemar significance tests
print(f"\n{'='*60}")
print(f"STEP 5: McNemar Significance Tests")
print(f"{'='*60}")

# Load zero-shot predictions from transfer evaluation
transfer_eval = json.loads((OUTPUT_DIR / 'metadata/transfer_evaluation.json').read_text())

# Get zero-shot predictions (re-evaluate)
ml_full_preds = np.array(ml_results[-1]['predictions'])
ml_baseline_full_preds = np.array(ml_baseline_results[-1]['predictions'])
zero_shot_preds = ml_model.predict(x_test_scaled)

# McNemar test: Fine-tuning vs Zero-shot
mcnemar_ft_vs_zs = transfer.mcnemar_test(
    y_test,
    zero_shot_preds,
    ml_full_preds,
)

# McNemar test: Fine-tuning vs From-scratch
mcnemar_ft_vs_fs = transfer.mcnemar_test(
    y_test,
    ml_baseline_full_preds,
    ml_full_preds,
)

print(f"\nMcNemar: Fine-tuning vs Zero-shot")
print(f"  Statistic: {mcnemar_ft_vs_zs['statistic']:.2f}")
print(f"  p-value: {mcnemar_ft_vs_zs['p_value']:.4f}")
print(f"  Significant: {mcnemar_ft_vs_zs['significant']}")

print(f"\nMcNemar: Fine-tuning vs From-scratch")
print(f"  Statistic: {mcnemar_ft_vs_fs['statistic']:.2f}")
print(f"  p-value: {mcnemar_ft_vs_fs['p_value']:.4f}")
print(f"  Significant: {mcnemar_ft_vs_fs['significant']}")

In [ ]:
# Step 6: Power-law curve fitting
print(f"\n{'='*60}")
print(f"STEP 6: Power-Law Curve Fitting")
print(f"{'='*60}")

# Prepare data for curve fitting
ml_df = pd.DataFrame(ml_results)
x_data = ml_df['n_samples'].values
y_data = ml_df['f1_score'].values

# Get target F1 (95% of Berlin performance)
berlin_eval = json.loads((OUTPUT_DIR / 'metadata/berlin_evaluation.json').read_text())
berlin_f1 = berlin_eval['metrics']['f1_score']
target_f1 = 0.95 * berlin_f1

# Fit power law
power_law_fit = evaluation.fit_power_law(
    x_data,
    y_data,
    target_y=target_f1,
)

print(f"\nPower-law fit: y = {power_law_fit['a']:.4f} * x^{power_law_fit['b']:.4f}")
print(f"R²: {power_law_fit['r_squared']:.4f}")
print(f"\nBerlin F1: {berlin_f1:.4f}")
print(f"Target (95%): {target_f1:.4f}")

if power_law_fit['extrapolated_x'] is not None:
    print(f"Estimated samples for 95% recovery: {power_law_fit['extrapolated_x']:.0f}")
    total_available = len(leipzig_finetune)
    print(f"Available Leipzig samples: {total_available}")
    if power_law_fit['extrapolated_x'] > total_available:
        print(f"  WARNING: Requires more samples than available")
else:
    print(f"95% recovery not achievable (already exceeded target)")

In [ ]:
# Step 7: Per-genus recovery analysis
print(f"\n{'='*60}")
print(f"STEP 7: Per-Genus Recovery Analysis")
print(f"{'='*60}")

# Compute per-genus metrics for each fraction
per_genus_recovery = []

for result in ml_results:
    preds = np.array(result['predictions'])
    per_class = evaluation.compute_per_class_metrics(y_test, preds, class_labels)
    
    for genus_metrics in per_class:
        per_genus_recovery.append({
            'fraction': result['fraction'],
            'genus': genus_metrics['genus'],
            'f1_score': genus_metrics['f1_score'],
            'support': genus_metrics['support'],
        })

per_genus_df = pd.DataFrame(per_genus_recovery)

# Show top 5 best and worst recovering genera
full_fraction_data = per_genus_df[per_genus_df['fraction'] == fractions[-1]]
top_5 = full_fraction_data.nlargest(5, 'f1_score')
bottom_5 = full_fraction_data.nsmallest(5, 'f1_score')

print(f"\nTop 5 best recovering genera (at full fine-tuning):")
for _, row in top_5.iterrows():
    print(f"  {row['genus']}: F1 = {row['f1_score']:.4f}")

print(f"\nTop 5 worst recovering genera (at full fine-tuning):")
for _, row in bottom_5.iterrows():
    print(f"  {row['genus']}: F1 = {row['f1_score']:.4f}")

In [ ]:
# Step 8: Visualization
print(f"\n{'='*60}")
print(f"STEP 8: Generate Figures")
print(f"{'='*60}")

fig_dir = OUTPUT_DIR / 'figures/finetuning'
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. Fine-tuning curves (ML with baselines)
baselines = {
    'zero_shot': transfer_eval['zero_shot_metrics']['f1_score'],
    'from_scratch_full': ml_baseline_results[-1]['f1_score'],
}

visualization.plot_finetuning_curve(
    ml_results,
    baselines=baselines,
    output_path=fig_dir / 'ml_finetuning_curve.png',
)

# 2. ML vs NN comparison
visualization.plot_finetuning_ml_vs_nn(
    ml_results,
    nn_results,
    output_path=fig_dir / 'ml_vs_nn_finetuning.png',
)

# 3. Per-genus recovery heatmap
# Transform per_genus_df to dict[float, dict[str, float]]
per_genus_by_fraction = {}
for frac in fractions:
    frac_data = per_genus_df[per_genus_df['fraction'] == frac]
    per_genus_by_fraction[frac] = dict(zip(frac_data['genus'], frac_data['f1_score']))

visualization.plot_finetuning_per_genus_recovery(
    per_genus_by_fraction,
    class_labels,
    output_path=fig_dir / 'per_genus_recovery.png',
)

print(f"\nFigures saved to {fig_dir}")

In [ ]:
# Save fine-tuning results
finetuning_data = {
    'ml_champion': ml_name,
    'nn_champion': nn_name,
    'fractions': fractions,
    'ml_results': ml_results,
    'nn_results': nn_results,
    'ml_baseline_results': ml_baseline_results,
    'mcnemar_tests': {
        'finetuning_vs_zeroshot': mcnemar_ft_vs_zs,
        'finetuning_vs_fromscratch': mcnemar_ft_vs_fs,
    },
    'power_law_fit': power_law_fit,
    'per_genus_recovery': per_genus_recovery,
}

curve_path = OUTPUT_DIR / 'metadata/finetuning_curve.json'
curve_path.write_text(json.dumps(finetuning_data, indent=2))

validate_finetuning_curve(curve_path)
print(f"\nSaved {curve_path}")

In [ ]:
# Execution log
elapsed = time.time() - start_time

log = {
    'status': 'completed',
    'timestamp': datetime.datetime.now().isoformat(),
    'runtime_seconds': elapsed,
    'config_hash': hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest(),
    'ml_final_f1': ml_results[-1]['f1_score'],
    'nn_final_f1': nn_results[-1]['f1_score'],
    'baseline_final_f1': ml_baseline_results[-1]['f1_score'],
    'power_law': {
        'a': power_law_fit['a'],
        'b': power_law_fit['b'],
        'r_squared': power_law_fit['r_squared'],
        'extrapolated_x': power_law_fit['extrapolated_x'],
    },
    'mcnemar_significant': {
        'vs_zeroshot': mcnemar_ft_vs_zs['significant'],
        'vs_fromscratch': mcnemar_ft_vs_fs['significant'],
    },
}

log_path = OUTPUT_DIR / 'logs/03d_finetuning.json'
log_path.write_text(json.dumps(log, indent=2))

print(f"\n{'='*60}")
print(f"FINE-TUNING COMPLETED")
print(f"{'='*60}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"ML final F1: {ml_results[-1]['f1_score']:.4f}")
print(f"NN final F1: {nn_results[-1]['f1_score']:.4f}")
print(f"Baseline F1: {ml_baseline_results[-1]['f1_score']:.4f}")
print(f"Output: {curve_path}")